# R11-H109 - the knockout matrix: how fragile is ceiling recall?

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R11 retrieval-correctness round <br>
**Graph**: rebuilt CPAP graph (neo4j2, read-only), Titan embeddings, zero completions <br>

Full-pipeline recall 1.0 is carried by overlapping channels. This ablates each channel and measures
how many golds survive on exactly one channel - single points of failure invisible at the ceiling.

## Approach (as registered, with the H108 correction)
1. **Assemble** the H34 full-context per probe: vec-8 seed renders + alias merge + prop_text channel
   + prop_node channel (4 channels)
2. **Independent presence** - for each gold string, `present()` is evaluated against EACH channel's
   assembled context text (the H108 correction: presence in assembled text, not per-seed placement),
   giving a gold x channel binary matrix
3. **Knockout** - ablating channel C loses exactly the golds present ONLY in C; a gold is
   SINGLE-CHANNEL if present in exactly one of the four channels
4. **Verdict** - >= 25% of golds single-channel confirms fragility; near-uniform multi-channel refutes

## Outputs
- `reports/knockout-matrix-h109-<stamp>.json`


In [1]:
# Imports
# stdlib
import datetime, json, os, re
from pathlib import Path
# third party
import yaml
from rich import print as rprint
from rich.progress import Progress

os.environ["NEO4J_URI"] = "bolt://user-konrad.jelen-kgf-neo4j2:7687"   # read-only neo4j2
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "kgfoundry"

# project
from knowledge_graph_foundry import Foundry, load_settings
from knowledge_graph_foundry.extraction import generate_embeddings
from knowledge_graph_foundry.graph.graphrag import vector_query
from knowledge_graph_foundry.graph.propositions import proposition_query
from knowledge_graph_foundry.models import Entity

PROBES_PATH = Path("../tests/probes/cpap-probe-set.yml")
settings = load_settings(Path("../config.yml"))
settings.graphrag.propositions_enabled = True
settings.graphrag.abstention_enabled = False
TOP_K = settings.graphrag.top_k                        # vec-8
PROP_TOP_K = settings.graphrag.proposition_top_k
VEC_INDEX = settings.graphrag.vector_index_name
PROP_INDEX = settings.graphrag.proposition_index_name
CHANNELS = ["vec", "alias", "prop_text", "prop_node"]
BAR_SINGLE = 0.25

probes = yaml.safe_load(PROBES_PATH.read_text())
gold_probes = [p for p in probes if p.get("gold_evidence")]
n_golds = sum(len(p["gold_evidence"]) for p in gold_probes)
rprint(f"[bold]config[/bold] channels={CHANNELS}  vec_top_k={TOP_K}  prop_top_k={PROP_TOP_K}  "
       f"golds={n_golds}  bar single-channel>={BAR_SINGLE}")


2026-07-07 12:18:50.322 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


config channels=['vec', 'alias', 'prop_text', 'prop_node']  vec_top_k=8  prop_top_k=8  golds=33  bar 
single-channel>=0.25

## Matcher (H34 iteration-3, verbatim) and per-node render\n\n`present()` and `render_nodes` copied verbatim from the H34 harness so channel contexts and the gold matcher are identical to production attribution.

In [2]:
def _norm(s):
    return re.sub(r"\s+", " ", s.casefold())

def value_tokens(text):
    return re.findall(r"[\w.\-/]*\d[\w.\-/]*", text)

_UNIT = r"(?<=\d)\s*(mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|mins|min|m)\b"

def present(gold, ctx_norm):
    ng = _norm(gold)
    if ng in ctx_norm:
        return True
    squashed = re.sub(r"[\s,()]", "", ctx_norm)
    skeleton = re.sub(r"[\s,()]", "", re.sub(_UNIT, "", ng))
    if any(ch.isdigit() for ch in skeleton) and len(skeleton) >= 5 and skeleton in squashed:
        return True
    tokens = value_tokens(gold)
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ctx_norm or re.sub(r"[\s,()]", "", _norm(t)) in squashed)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng))
    ctx_words = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx_norm))
    return bool(words) and len(words & ctx_words) / len(words) >= 0.6

def render_nodes(session, node_ids, include_alias=True):
    blocks = []
    for nid in node_ids:
        row = session.run(
            "MATCH (e:Entity {id: $id}) RETURN e.name AS name, labels(e) AS types, "
            "e.description AS description, properties(e) AS props", id=nid).single()
        if row is None:
            continue
        spec = {k.removeprefix("prop_"): v for k, v in row["props"].items() if k.startswith("prop_")}
        alias_names = []
        if include_alias:
            aliases = session.run(
                "MATCH (e:Entity {id: $id})-[:SAME_AS*1..2]-(a:Entity) "
                "WHERE a.id <> $id RETURN DISTINCT a.name AS name, properties(a) AS props LIMIT 5", id=nid).data()
            alias_names = [a["name"] for a in aliases]
            for a in aliases:
                for k, v in a["props"].items():
                    if k.startswith("prop_"):
                        spec.setdefault(k.removeprefix("prop_"), v)
        rels = session.run(
            "MATCH (e:Entity {id: $id})-[r]-(n:Entity) "
            "WHERE r.valid_to IS NULL AND type(r) <> 'SIMILAR_TO' "
            "RETURN type(r) AS rel, n.name AS name LIMIT 15", id=nid).data()
        blocks.append(
            f"## {row['name']} ({', '.join(row['types'])})\n"
            + (f"Also known as: {', '.join(alias_names)}\n" if alias_names else "")
            + f"{row['description'] or ''}\n"
            f"Properties: {json.dumps(spec, default=str)}\n"
            "Relations: " + "; ".join(f"{r['rel']} -> {r['name']}" for r in rels))
    return "\n".join(blocks)


## Channel harvest\n\nPer probe: one Titan embedding, then the four channels assembled separately with production parameters. The `alias` channel isolates the alias contribution by differencing the alias-merged render against the no-alias render (so `alias` = strings the alias merge ADDS, not the seed's own facts).

In [3]:
harvest = {}
with Foundry(settings) as f, Progress() as pr:
    t = pr.add_task("channel harvest", total=len(gold_probes))
    for p in gold_probes:
        q = p["question"]
        probe_e = Entity.create(q[:80], types=["Query"], description=q)
        emb = generate_embeddings([probe_e], settings.embeddings)[0].embedding
        seeds = vector_query(f.driver, emb, VEC_INDEX, top_k=TOP_K)
        seed_ids = [s["id"] for s in seeds]
        prop_hits = proposition_query(f.driver, emb, PROP_INDEX, top_k=PROP_TOP_K)
        prop_seed_ids = [eid for h in prop_hits for eid in h["entity_ids"] if eid not in seed_ids]
        with f.driver.session() as session:
            ctx_vec = render_nodes(session, seed_ids, include_alias=False)
            ctx_vec_alias = render_nodes(session, seed_ids, include_alias=True)
            ctx_prop_node = render_nodes(session, prop_seed_ids, include_alias=True)
        ctx_prop_text = "\n".join(h["text"] for h in prop_hits)
        harvest[p["id"]] = {"gold": p["gold_evidence"], "ctx": {
            "vec": _norm(ctx_vec),
            "alias": _norm(ctx_vec_alias),   # alias-merged render (superset of vec)
            "prop_text": _norm(ctx_prop_text),
            "prop_node": _norm(ctx_prop_node),
        }}
        pr.advance(t)
rprint(f"[green]harvested[/green] {len(harvest)} probes x {len(CHANNELS)} channels")


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-07-07 12:18:51.123 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 
1/1

2026-07-07 12:18:51.125 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - 
Generated embeddings for 1/1 entities via bedrock (0 cache hits)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

...

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key
does not exist. The property `valid_to` does not exist. Verify that the spelling is correct.', 
position=<SummaryInputPosition line=1, column=51, offset=50>, raw_classification='UNRECOGNIZED', 
classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', 
severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', 
'_severity': 'WARNING', '_position': {'offset': 50, 'line': 1, 'column': 51}, 'OPERATION': '', 'OPERATION_CODE': 
'0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (e:Entity {id: $id})-[r]-(n:Entity) WHERE r.valid_to IS NULL AND 
type(r) <> 'SIMILAR_TO' RETURN type(r) AS rel, n.name AS name LIMIT 15"

harvested 24 probes x 4 channels

## Gold x channel matrix\n\nEach gold's presence is evaluated against each channel independently. The `alias` cell is TRUE only when the alias merge adds the gold that the seed's own render (`vec`) lacks - isolating the exclusive alias contribution. A gold present in exactly one channel is single-channel (one knockout from loss).

In [4]:
matrix = []   # per gold: presence per channel (alias isolated)
for pid, h in harvest.items():
    ctx = h["ctx"]
    for g in h["gold"]:
        in_vec = present(g, ctx["vec"])
        in_alias_render = present(g, ctx["alias"])
        cells_ = {
            "vec": in_vec,
            "alias": bool(in_alias_render and not in_vec),   # alias-exclusive contribution
            "prop_text": present(g, ctx["prop_text"]),
            "prop_node": present(g, ctx["prop_node"]),
        }
        present_channels = [c for c in CHANNELS if cells_[c]]
        matrix.append({"probe": pid, "gold": g[:70], "cells": cells_,
                       "channels": present_channels, "n_channels": len(present_channels)})

surfaced = [m for m in matrix if m["n_channels"] >= 1]
missing = [m for m in matrix if m["n_channels"] == 0]
single = [m for m in surfaced if m["n_channels"] == 1]
multi = [m for m in surfaced if m["n_channels"] >= 2]

# per-channel exclusivity (golds lost if that channel is knocked out)
exclusivity = {c: sum(1 for m in single if m["channels"][0] == c) for c in CHANNELS}
# per-channel total carriage
carriage = {c: sum(1 for m in surfaced if m["cells"][c]) for c in CHANNELS}

n_total = len(matrix)
single_share_all = len(single) / n_total if n_total else 0.0
single_share_surf = len(single) / len(surfaced) if surfaced else 0.0
rprint(f"golds total [yellow]{n_total}[/yellow]  surfaced [green]{len(surfaced)}[/green]  "
       f"missing [red]{len(missing)}[/red]")
rprint(f"single-channel golds [bold yellow]{len(single)}[/bold yellow]  multi-channel [yellow]{len(multi)}[/yellow]")
rprint(f"single-channel share: of-all [bold]{single_share_all:.3f}[/bold]  of-surfaced [bold]{single_share_surf:.3f}[/bold] "
       f"[dim](bar >= {BAR_SINGLE})[/dim]")
rprint(f"per-channel carriage: {carriage}")
rprint(f"per-channel exclusivity (knockout loss): {exclusivity}")


golds total 33  surfaced 33  missing 0

single-channel golds 20  multi-channel 13

single-channel share: of-all 0.606  of-surfaced 0.606 (bar >= 0.25)

per-channel carriage: {'vec': 21, 'alias': 2, 'prop_text': 11, 'prop_node': 21}

per-channel exclusivity (knockout loss): {'vec': 8, 'alias': 2, 'prop_text': 2, 'prop_node': 8}

## Matrix render, verdict, report\n\nThe verdict reads on the single-channel share of surfaced golds. H108's refutation (no render fix ships; the global proposition channel is the correct mechanism, not redundancy) means the registered post-H108 halving sub-clause does not apply - reported for the record.

In [5]:
glyph = {True: "#", False: "."}
rprint(f"[bold cyan]Gold x channel knockout matrix[/bold cyan] [dim](# present, . absent; alias=exclusive)[/dim]")
rprint(f"  {'probe/gold':40s} " + " ".join(f"{c[:4]:>4s}" for c in CHANNELS) + "  n")
for m in matrix:
    line = " ".join(f"{glyph[m['cells'][c]]:>4s}" for c in CHANNELS)
    tag = "[red]MISS[/red]" if m["n_channels"] == 0 else ("[yellow]SINGLE[/yellow]" if m["n_channels"] == 1 else "")
    rprint(f"  [dim]{m['probe']}[/dim] {m['gold'][:33]:33s} {line}  {m['n_channels']} {tag}")

verdict = ("CONFIRMED" if single_share_surf >= BAR_SINGLE else "REFUTED")
n_by_count = {k: sum(1 for m in surfaced if m["n_channels"] == k) for k in (1, 2, 3, 4)}
rprint(f"""
[bold cyan]Verdict[/bold cyan]
[dim]{"-"*44}[/dim]
  channel-count distribution (surfaced): {n_by_count}
  single-channel share of surfaced: [bold yellow]{single_share_surf:.3f}[/bold yellow] [dim](bar >= {BAR_SINGLE})[/dim]
  Verdict: [{'green' if verdict=='CONFIRMED' else 'red'}]{verdict}[/]
  [dim]H108 post-fix halving sub-clause: N/A (H108 refuted, no render fix ships)[/dim]
""")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"knockout-matrix-h109-{stamp}.json"
out.write_text(json.dumps({
    "hypothesis": "R11-H109", "channels": CHANNELS,
    "n_golds": n_total, "n_surfaced": len(surfaced), "n_missing": len(missing),
    "n_single_channel": len(single), "n_multi_channel": len(multi),
    "single_share_of_all": single_share_all, "single_share_of_surfaced": single_share_surf,
    "channel_count_distribution": n_by_count,
    "per_channel_carriage": carriage, "per_channel_exclusivity": exclusivity,
    "bar_single_channel": BAR_SINGLE, "verdict": verdict,
    "matrix": matrix,
}, indent=2, default=str))
rprint("saved", str(out))


Gold x channel knockout matrix (# present, . absent; alias=exclusive)

probe/gold                                vec alia prop prop  n

P01 1130 g                               .    .    .    #  1 SINGLE

P01 40 oz                                .    .    .    #  1 SINGLE

P02 4 to 20 cm H2O                       #    .    #    #  3

...

P22 amplitude of oscillations            .    .    #    .  1 SINGLE

P23 sleep onset detection                #    .    #    #  3

P24 reduces the pressure during expir    #    .    .    .  1 SINGLE

Verdict
--------------------------------------------
  channel-count distribution (surfaced): {1: 20, 2: 4, 3: 9, 4: 0}
  single-channel share of surfaced: 0.606 (bar >= 0.25)
  Verdict: CONFIRMED
  H108 post-fix halving sub-clause: N/A (H108 refuted, no render fix ships)

saved ../reports/knockout-matrix-h109-20260707-101900.json